In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install playwright pandas pyarrow pillow numpy huggingface_hub
!playwright install --with-deps chromium firefox webkit


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Installing dependencies...
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:3 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libasound2t64 

In [ ]:
"""
Design2Code -> paired visual-regression dataset (mutations + cross-browser benign), one script.

For every page, rendered with ONE fixed set of settings
(viewport 1280x800, full_page, all network blocked, rick.jpg inlined, wait "load" + 150 ms):

  d2c_XXX.png                  reference (expected) render, Chromium
  d2c_XXX_<operator>.png       single-element CSS mutation, Chromium   -> label "regression"
  d2c_XXX_identical.png        same HTML re-rendered, Chromium          -> label "no_change"
  d2c_XXX_firefox.png          same HTML in Firefox                     -> label "benign"
  d2c_XXX_webkit.png           same HTML in WebKit                      -> label "benign"

Every candidate is paired with the SAME Chromium reference image.

Outputs (in OUT_DIR):
  screenshots/                     all images
  metadata/d2c_XXX.json            all records of one page (used for resume)
  master_dataset.json              regression + no_change only   (experiment A)
  master_dataset_with_benign.json  regression + no_change + benign (experiment B)

Resume: pages that already have metadata are skipped; if they are missing benign
renders (e.g. produced by an older run), only the missing browsers are rendered.

Colab usage (CPU runtime is fine):
    !pip -q install playwright pandas pyarrow pillow numpy huggingface_hub
    !playwright install --with-deps chromium firefox webkit
    (paste this file into a cell and run it, then in the next cell:)
    END_INDEX = 5      # quick test; set back to None for the full run
    await main()
"""

import os
import json
import random
import asyncio
import colorsys
import re

import numpy as np
import pandas as pd
from PIL import Image
from playwright.async_api import async_playwright

# ==========================================================
# PLACEHOLDER IMAGE (inline rick.jpg so every engine draws a real image)
# ==========================================================
import re
import io
import base64


def _load_placeholder_uri():
    """Return rick.jpg as a data URI; fall back to a neutral grey JPEG if the download fails."""
    try:
        from huggingface_hub import hf_hub_download
        path = hf_hub_download("SALT-NLP/Design2Code", "rick.jpg", repo_type="dataset")
        data = open(path, "rb").read()
        print("placeholder: rick.jpg from SALT-NLP/Design2Code")
    except Exception as e:
        print("placeholder: rick.jpg download failed, using grey JPEG instead:", str(e)[:80])
        buf = io.BytesIO()
        Image.new("RGB", (640, 480), (200, 200, 200)).save(buf, format="JPEG", quality=90)
        data = buf.getvalue()
    return "data:image/jpeg;base64," + base64.b64encode(data).decode()


PLACEHOLDER_URI = _load_placeholder_uri()
_SRC_RE = re.compile(r"""(src\s*=\s*)(["']?)rick\.jpg\2""", re.IGNORECASE)
_URL_RE = re.compile(r"""url\(\s*(["']?)rick\.jpg\1\s*\)""", re.IGNORECASE)


def prep_html(html):
    """Replace every reference to rick.jpg (img src and CSS url()) with the inlined data URI."""
    html = _SRC_RE.sub(lambda m: f'{m.group(1)}"{PLACEHOLDER_URI}"', html)
    # unquoted url(): safe inside inline style="..." attributes (base64 has no spaces or parens)
    html = _URL_RE.sub(lambda m: f"url({PLACEHOLDER_URI})", html)
    return html

# ==========================================================
# CONFIG
# ==========================================================
PARQUET_PATH = "/content/drive/MyDrive/CSC490_data/train-00000-of-00001.parquet"
OUT_DIR = "/content/drive/MyDrive/csc490/d2c_v2"   # on Drive so a disconnect does not lose work
SEED = 490

VIEWPORT = {"width": 1280, "height": 800}
SETTLE_MS = 150              # wait after load / after applying a mutation
LOAD_TIMEOUT_MS = 15000

OPS_PER_PAGE = 13            # how many operators to sample per page (13 = all of them)
MAX_TRIES_PER_OP = 8         # retries with a different target element if a sample is rejected
MAGNITUDES = ["small", "medium", "large"]

MAKE_IDENTICAL = True        # same-engine re-render  -> "no_change"
BENIGN_BROWSERS = ["firefox", "webkit"]   # non-Blink engines -> label "benign"

MIN_CHANGED_PX = 60          # fewer changed pixels than this -> reject (invisible mutation)
MAX_LOCAL_RATIO = 6.0        # non-layout ops: diff bbox area / target bbox area above this -> reject
BBOX_PAD = 12                # padding (px) around the target bbox when checking locality

START_INDEX = 0              # set to process a subset, e.g. for a quick test
END_INDEX = None             # e.g. 5 for a quick test run

# ==========================================================
# OPERATORS (13, grouped into families)
# ==========================================================
TEXT_TAGS = "p,h1,h2,h3,h4,h5,h6,a,span,li,button,label,td,th,div"
BOX_TAGS = "div,section,header,footer,nav,article,aside,form,button,a,img,li,input"

OPERATORS = {
    # typography
    "font_size":        {"family": "typography", "prop": "font-size",        "tags": TEXT_TAGS, "text": True},
    "font_weight":      {"family": "typography", "prop": "font-weight",      "tags": TEXT_TAGS, "text": True},
    "line_height":      {"family": "typography", "prop": "line-height",      "tags": TEXT_TAGS, "text": True},
    "letter_spacing":   {"family": "typography", "prop": "letter-spacing",   "tags": TEXT_TAGS, "text": True},
    # colour
    "text_color":       {"family": "colour",     "prop": "color",            "tags": TEXT_TAGS, "text": True},
    "background_color": {"family": "colour",     "prop": "background-color", "tags": BOX_TAGS,  "text": False},
    # appearance
    "opacity":          {"family": "appearance", "prop": "opacity",          "tags": BOX_TAGS,  "text": False},
    # shape / chrome
    "border_radius":    {"family": "shape",      "prop": "border-radius",    "tags": BOX_TAGS,  "text": False},
    "border_width":     {"family": "shape",      "prop": "border",           "tags": BOX_TAGS,  "text": False},
    # layout
    "padding":          {"family": "layout",     "prop": "padding",          "tags": BOX_TAGS,  "text": False},
    "margin":           {"family": "layout",     "prop": "margin",           "tags": BOX_TAGS,  "text": False},
    "position":         {"family": "layout",     "prop": "transform",        "tags": BOX_TAGS,  "text": False},
    "width":            {"family": "layout",     "prop": "width",            "tags": BOX_TAGS,  "text": False},
}
LAYOUT_FAMILY = "layout"     # ops in this family may legitimately reflow the page

# ==========================================================
# JS HELPERS (run inside the page)
# ==========================================================
JS_PICK_TARGET = """
(args) => {
  const {tags, needText, r} = args;
  const docW = document.documentElement.scrollWidth;
  const docH = document.documentElement.scrollHeight;
  const ok = [];
  for (const el of document.querySelectorAll(tags)) {
    const rect = el.getBoundingClientRect();
    if (rect.width < 12 || rect.height < 8) continue;
    const x = rect.left + window.scrollX, y = rect.top + window.scrollY;
    if (x < 0 || y < 0 || x + rect.width > docW + 1 || y + rect.height > docH + 1) continue;
    if (rect.width * rect.height > 0.25 * docW * docH) continue;     // avoid page-sized containers
    let visible = true;
    for (let a = el; a; a = a.parentElement) {
      const cs = getComputedStyle(a);
      if (cs.display === 'none' || cs.visibility === 'hidden' || parseFloat(cs.opacity) < 0.1) { visible = false; break; }
    }
    if (!visible) continue;
    if (needText) {
      // require direct text so the change lands on visible glyphs
      const direct = Array.from(el.childNodes).some(n => n.nodeType === 3 && n.textContent.trim().length > 1);
      if (!direct) continue;
    }
    ok.push(el);
  }
  if (!ok.length) return null;
  const el = ok[Math.floor(r * ok.length)];
  document.querySelectorAll('[data-mut-target]').forEach(e => e.removeAttribute('data-mut-target'));
  el.setAttribute('data-mut-target', '1');

  // readable CSS path for metadata
  const parts = [];
  for (let a = el; a && a.nodeType === 1 && a !== document.documentElement; a = a.parentElement) {
    let s = a.tagName.toLowerCase();
    const p = a.parentElement;
    if (p) {
      const same = Array.from(p.children).filter(c => c.tagName === a.tagName);
      if (same.length > 1) s += `:nth-of-type(${same.indexOf(a) + 1})`;
    }
    parts.unshift(s);
  }
  return {selector: parts.join(' > '), tag: el.tagName.toLowerCase(),
          text: (el.innerText || '').trim().slice(0, 80), candidates: ok.length};
}
"""

JS_READ_STYLE = """
() => {
  const el = document.querySelector('[data-mut-target]');
  const cs = getComputedStyle(el);
  const keys = ['font-size','font-weight','line-height','letter-spacing','color','background-color',
                'opacity','border-top-left-radius','border-top-width','border-top-style',
                'padding-top','padding-right','padding-bottom','padding-left',
                'margin-top','margin-left','width','transform','display'];
  const out = {};
  for (const k of keys) out[k] = cs.getPropertyValue(k);
  // effective background: first non-transparent background among element and ancestors
  let eff = 'rgb(255, 255, 255)';
  for (let a = el; a; a = a.parentElement) {
    const bg = getComputedStyle(a).backgroundColor;
    const m = bg.match(/[\\d.]+/g);
    if (m && (m.length < 4 || parseFloat(m[3]) > 0)) { eff = bg; break; }
  }
  out['effective-bg'] = eff;
  out['tag'] = el.tagName.toLowerCase();
  const own = (cs.backgroundColor.match(/[\\d.]+/g) || []);
  const ownOpaque = own.length >= 3 && (own.length < 4 || parseFloat(own[3]) > 0);
  out['has-box'] = ownOpaque ||
                   (parseFloat(cs.borderTopWidth) > 0 && cs.borderTopStyle !== 'none') ||
                   (cs.backgroundImage && cs.backgroundImage !== 'none') ||
                   (cs.boxShadow && cs.boxShadow !== 'none');
  return out;
}
"""

JS_APPLY = """
(decls) => {
  const el = document.querySelector('[data-mut-target]');
  for (const [k, v] of Object.entries(decls)) el.style.setProperty(k, v, 'important');
}
"""

JS_BBOX = """
() => {
  const el = document.querySelector('[data-mut-target]');
  const r = el.getBoundingClientRect();
  return [Math.round(r.left + window.scrollX), Math.round(r.top + window.scrollY),
          Math.round(r.width), Math.round(r.height)];
}
"""

# ==========================================================
# VALUE GENERATION (python side, reproducible via seeded RNG)
# ==========================================================
def px(v, default=0.0):
    m = re.match(r"^\s*(-?[\d.]+)px", v or "")
    return float(m.group(1)) if m else default

def parse_rgba(v):
    nums = [float(x) for x in re.findall(r"[\d.]+", v or "")]
    if len(nums) < 3:
        return (0, 0, 0, 0.0)
    a = nums[3] if len(nums) >= 4 else 1.0
    return (nums[0], nums[1], nums[2], a)

def shift_colour(v, mag, rng, fallback="rgb(255, 255, 255)"):
    r, g, b, a = parse_rgba(v)
    if a == 0:                                   # transparent: start from the colour actually visible behind it
        r, g, b, _ = parse_rgba(fallback)
        a = 1.0
    h, l, s = colorsys.rgb_to_hls(r / 255, g / 255, b / 255)
    dh = {"small": 0.04, "medium": 0.12, "large": 0.30}[mag]
    dl = {"small": 0.06, "medium": 0.14, "large": 0.28}[mag]
    h = (h + rng.choice([-1, 1]) * dh) % 1.0
    s = min(1.0, max(s, 0.35))                   # make hue shifts visible on greys
    l = l + dl if l < 0.5 else l - dl
    l = min(0.95, max(0.05, l))
    nr, ng, nb = colorsys.hls_to_rgb(h, l, s)
    return f"rgba({round(nr*255)}, {round(ng*255)}, {round(nb*255)}, {a})"

def make_decls(op, style, mag, rng):
    """Return (css declarations to apply, old_value, new_value)."""
    sign = rng.choice([-1, 1])
    f = {"small": 1.12, "medium": 1.3, "large": 1.6}[mag]
    d = {"small": 3, "medium": 8, "large": 20}[mag]

    if op == "font_size":
        old = px(style["font-size"], 16)
        new = old * (f if sign > 0 else 1 / f)
        return {"font-size": f"{new:.1f}px"}, style["font-size"], f"{new:.1f}px"
    if op == "font_weight":
        old = int(float(style["font-weight"] or 400))
        step = {"small": 100, "medium": 200, "large": 400}[mag]
        new = old + step if old + step <= 900 else old - step
        new = min(900, max(100, new))
        if new == old:
            return None
        return {"font-weight": str(new)}, str(old), str(new)
    if op == "line_height":
        fs = px(style["font-size"], 16)
        old = px(style["line-height"], fs * 1.2)
        new = old * (f if sign > 0 else 1 / f)
        return {"line-height": f"{new:.1f}px"}, style["line-height"], f"{new:.1f}px"
    if op == "letter_spacing":
        old = px(style["letter-spacing"], 0.0)
        delta = {"small": 0.6, "medium": 1.5, "large": 3.5}[mag]
        new = old + sign * delta
        return {"letter-spacing": f"{new:.2f}px"}, style["letter-spacing"], f"{new:.2f}px"
    if op == "text_color":
        new = shift_colour(style["color"], mag, rng)
        return {"color": new}, style["color"], new
    if op == "background_color":
        new = shift_colour(style["background-color"], mag, rng, fallback=style["effective-bg"])
        return {"background-color": new}, style["background-color"], new
    if op == "opacity":
        old = float(style["opacity"] or 1)
        new = old * (1 - {"small": 0.15, "medium": 0.35, "large": 0.65}[mag])
        return {"opacity": f"{new:.2f}"}, style["opacity"], f"{new:.2f}"
    if op == "border_radius":
        if not style["has-box"]:
            return None                         # no background/border: rounded corners are invisible
        old = px(style["border-top-left-radius"], 0)
        new = old + {"small": 4, "medium": 10, "large": 24}[mag]
        return {"border-radius": f"{new:.0f}px"}, style["border-top-left-radius"], f"{new:.0f}px"
    if op == "border_width":
        w = {"small": 1, "medium": 2, "large": 4}[mag]
        old_w = px(style["border-top-width"], 0)
        if style["border-top-style"] not in ("none", "hidden") and old_w > 0:
            new_w = old_w + w
            return {"border-width": f"{new_w:.0f}px"}, style["border-top-width"], f"{new_w:.0f}px"
        return ({"border": f"{w}px solid rgba(0,0,0,0.6)"},
                "none", f"{w}px solid rgba(0,0,0,0.6)")
    if op == "padding":
        sides = ["padding-top", "padding-right", "padding-bottom", "padding-left"]
        decls = {k: f"{max(0, px(style[k]) + d):.0f}px" for k in sides}
        old = " ".join(style[k] for k in sides)
        return decls, old, " ".join(decls[k] for k in sides)
    if op == "margin":
        k = rng.choice(["margin-top", "margin-left"])
        new = px(style[k]) + d
        return {k: f"{new:.0f}px"}, f"{k}: {style[k]}", f"{k}: {new:.0f}px"
    if op == "position":
        dx, dy = rng.choice([(d, 0), (-d, 0), (0, d), (0, -d), (d, d)])
        base = style["transform"] if style["transform"] not in ("", "none") else ""
        new = f"{base} translate({dx}px, {dy}px)".strip()
        return {"transform": new}, style["transform"], new
    if op == "width":
        if style["display"].startswith("inline") and style["display"] != "inline-block":
            return None                         # width has no effect on inline elements
        old = px(style["width"], 0)
        if old <= 0:
            return None
        new = old * (1 / f)                     # shrink: growing often has no visible effect
        return {"width": f"{new:.0f}px", "box-sizing": "border-box"}, style["width"], f"{new:.0f}px"
    raise ValueError(op)

# ==========================================================
# IMAGE DIFF
# ==========================================================
def load_rgb(path):
    return np.asarray(Image.open(path).convert("RGB"), dtype=np.int16)

def diff_stats(ref_path, cand_path):
    a, b = load_rgb(ref_path), load_rgb(cand_path)
    H, W = max(a.shape[0], b.shape[0]), max(a.shape[1], b.shape[1])
    pa = np.full((H, W, 3), 255, np.int16); pa[:a.shape[0], :a.shape[1]] = a
    pb = np.full((H, W, 3), 255, np.int16); pb[:b.shape[0], :b.shape[1]] = b
    mask = (np.abs(pa - pb).max(axis=2) > 0)
    n = int(mask.sum())
    if n == 0:
        return {"changed_px": 0, "diff_bbox": None, "size_ref": list(a.shape[1::-1]), "size_cand": list(b.shape[1::-1])}
    ys, xs = np.where(mask)
    x0, y0, x1, y1 = int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1
    return {"changed_px": n, "diff_bbox": [x0, y0, x1 - x0, y1 - y0],
            "size_ref": list(a.shape[1::-1]), "size_cand": list(b.shape[1::-1])}

def union_bbox(b1, b2, pad):
    x0 = min(b1[0], b2[0]) - pad; y0 = min(b1[1], b2[1]) - pad
    x1 = max(b1[0] + b1[2], b2[0] + b2[2]) + pad; y1 = max(b1[1] + b1[3], b2[1] + b2[3]) + pad
    return [x0, y0, x1 - x0, y1 - y0]

# ==========================================================
# RENDERING
# ==========================================================
async def new_page(browser):
    page = await browser.new_page(viewport=VIEWPORT)
    page.set_default_timeout(LOAD_TIMEOUT_MS)
    # Block all network so every render is deterministic and "load" never hangs.
    # rick.jpg is inlined as a data: URI by prep_html(), so images still render.
    async def block(route):
        if route.request.url.startswith("data:"):
            await route.continue_()
        else:
            await route.abort()
    await page.route("**/*", block)
    return page

async def load(page, html):
    await page.set_content(prep_html(html), wait_until="load", timeout=LOAD_TIMEOUT_MS)
    await page.wait_for_timeout(SETTLE_MS)

async def shot(page, path):
    await page.screenshot(path=path, full_page=True)

# ==========================================================
# PER-PAGE PIPELINE
# ==========================================================
async def process_page(idx, html, chromium, benign_browsers, rng, dirs):
    pid = f"d2c_{idx:03d}"
    img = lambda name: os.path.join(dirs["img"], f"{name}.png")
    rel = lambda name: f"screenshots/{name}.png"
    records = []

    page = await new_page(chromium)
    try:
        # ---- reference ----
        await load(page, html)
        await shot(page, img(pid))
        ref = img(pid)

        # ---- identical re-render (no_change) ----
        if MAKE_IDENTICAL:
            await load(page, html)
            await shot(page, img(f"{pid}_identical"))
            st = diff_stats(ref, img(f"{pid}_identical"))
            records.append({"sample_id": f"{pid}_identical", "parent_id": pid,
                            "reference_path": rel(pid), "candidate_path": rel(f"{pid}_identical"),
                            "label": "no_change", "family": None, "operator": None,
                            "browser_ref": "chromium", "browser_cand": "chromium", **st})

        # ---- mutations (regression) ----
        ops = list(OPERATORS)
        rng.shuffle(ops)
        for op in ops[:OPS_PER_PAGE]:
            spec = OPERATORS[op]
            sid = f"{pid}_{op}"
            for attempt in range(MAX_TRIES_PER_OP):
                mag = rng.choice(MAGNITUDES)
                await load(page, html)
                target = await page.evaluate(JS_PICK_TARGET,
                                             {"tags": spec["tags"], "needText": spec["text"], "r": rng.random()})
                if target is None:
                    break
                style = await page.evaluate(JS_READ_STYLE)
                made = make_decls(op, style, mag, rng)
                if made is None:
                    continue
                decls, old_v, new_v = made
                bbox_before = await page.evaluate(JS_BBOX)
                await page.evaluate(JS_APPLY, decls)
                await page.wait_for_timeout(SETTLE_MS)
                bbox_after = await page.evaluate(JS_BBOX)
                await shot(page, img(sid))

                st = diff_stats(ref, img(sid))
                if st["changed_px"] < MIN_CHANGED_PX:
                    continue                                    # invisible -> retry
                tb = union_bbox(bbox_before, bbox_after, BBOX_PAD)
                db = st["diff_bbox"]
                ratio = (db[2] * db[3]) / max(1, tb[2] * tb[3])
                reflow = ratio > MAX_LOCAL_RATIO
                if reflow and spec["family"] != LAYOUT_FAMILY:
                    continue                                    # non-local change -> retry

                records.append({
                    "sample_id": sid, "parent_id": pid,
                    "reference_path": rel(pid), "candidate_path": rel(sid),
                    "label": "regression", "family": spec["family"], "operator": op,
                    "css_property": spec["prop"], "old_value": old_v, "new_value": new_v,
                    "applied": decls, "magnitude": mag,
                    "target_selector": target["selector"], "target_tag": target["tag"],
                    "target_text": target["text"],
                    "bbox_before": bbox_before, "bbox_after": bbox_after,
                    "locality_ratio": round(ratio, 2), "reflow": reflow,
                    "attempt": attempt, "browser_ref": "chromium", "browser_cand": "chromium",
                    **st})
                break
            else:
                if os.path.exists(img(sid)):
                    os.remove(img(sid))                         # all attempts rejected
    finally:
        await page.close()

    # ---- cross-browser benign (same HTML, different engine, same Chromium reference) ----
    records += await render_benign(pid, html, benign_browsers, dirs)

    return records

async def render_benign(pid, html, benign_browsers, dirs):
    """Render the unmodified page in each non-Chromium engine; one failing browser does not stop the others."""
    img = lambda name: os.path.join(dirs["img"], f"{name}.png")
    rel = lambda name: f"screenshots/{name}.png"
    ref = img(pid)
    out = []
    for name, browser in benign_browsers.items():
        sid = f"{pid}_{name}"
        bpage = await new_page(browser)
        try:
            await load(bpage, html)
            await shot(bpage, img(sid))
            st = diff_stats(ref, img(sid))
            out.append({"sample_id": sid, "parent_id": pid,
                        "reference_path": rel(pid), "candidate_path": rel(sid),
                        "label": "benign", "family": None, "operator": None,
                        "browser_ref": "chromium", "browser_cand": name,
                        "browser_cand_version": browser.version, **st})
        except Exception as e:
            print(f"    {sid}: benign render failed: {str(e)[:100]}")
        finally:
            await bpage.close()
    return out


# ==========================================================
# MAIN
# ==========================================================
async def main():
    dirs = {"img": os.path.join(OUT_DIR, "screenshots"), "meta": os.path.join(OUT_DIR, "metadata")}
    for d in dirs.values():
        os.makedirs(d, exist_ok=True)
    old_benign_dir = os.path.join(OUT_DIR, "metadata_benign")   # written by the older render_benign.py

    df = pd.read_parquet(PARQUET_PATH)
    end = len(df) if END_INDEX is None else min(END_INDEX, len(df))
    failed = []

    async with async_playwright() as p:
        chromium = await p.chromium.launch(headless=True)
        benign_browsers = {name: await getattr(p, name).launch(headless=True) for name in BENIGN_BROWSERS}
        print("chromium", chromium.version, "|", ", ".join(f"{n} {b.version}" for n, b in benign_browsers.items()))

        for idx in range(START_INDEX, end):
            pid = f"d2c_{idx:03d}"
            meta_path = os.path.join(dirs["meta"], f"{pid}.json")
            html = df.iloc[idx]["text"]
            if not isinstance(html, str) or not html.strip():
                continue

            # ---- resume: page already done -> only back-fill missing benign renders ----
            if os.path.exists(meta_path):
                with open(meta_path) as f:
                    recs = json.load(f)
                have = {r["browser_cand"] for r in recs if r["label"] == "benign"}
                old_path = os.path.join(old_benign_dir, f"{pid}.json")
                if os.path.exists(old_path):
                    with open(old_path) as f:
                        old = [r for r in json.load(f) if r["browser_cand"] not in have]
                    recs += old
                    have |= {r["browser_cand"] for r in old}
                missing = {n: b for n, b in benign_browsers.items() if n not in have}
                if missing and os.path.exists(os.path.join(dirs["img"], f"{pid}.png")):
                    recs += await render_benign(pid, html, missing, dirs)
                    print(f"[{idx + 1}/{end}] {pid}: back-filled benign for {sorted(missing)}")
                if missing or os.path.exists(old_path):
                    with open(meta_path, "w") as f:
                        json.dump(recs, f, indent=2)
                continue

            # ---- new page: reference + identical + mutations + benign ----
            rng = random.Random(SEED * 100000 + idx)            # per-page seed: reproducible + resumable
            try:
                recs = await process_page(idx, html, chromium, benign_browsers, rng, dirs)
                with open(meta_path, "w") as f:
                    json.dump(recs, f, indent=2)
                c = {k: sum(r["label"] == k for r in recs) for k in ["regression", "no_change", "benign"]}
                print(f"[{idx + 1}/{end}] {pid}: {c['regression']} regression, "
                      f"{c['no_change']} no_change, {c['benign']} benign")
            except Exception as e:
                failed.append((idx, str(e)[:120]))
                print(f"[{idx + 1}/{end}] {pid}: FAILED {str(e)[:120]}")

        await chromium.close()
        for b in benign_browsers.values():
            await b.close()

    # ---- master indexes ----
    all_recs = []
    for name in sorted(os.listdir(dirs["meta"])):
        with open(os.path.join(dirs["meta"], name)) as f:
            all_recs.extend(json.load(f))
    base = [r for r in all_recs if r["label"] != "benign"]
    with open(os.path.join(OUT_DIR, "master_dataset.json"), "w") as f:
        json.dump(base, f, indent=2)
    with open(os.path.join(OUT_DIR, "master_dataset_with_benign.json"), "w") as f:
        json.dump(all_recs, f, indent=2)

    by_label = {}
    for r in all_recs:
        by_label[r["label"]] = by_label.get(r["label"], 0) + 1
    print("\nDone. Samples by label:", by_label)
    print("Failed pages:", len(failed), failed[:5])
    benign = [r for r in all_recs if r["label"] == "benign"]
    if benign:
        size_changed = sum(r["size_ref"] != r["size_cand"] for r in benign)
        print(f"benign renders with a different page size: {size_changed}/{len(benign)}")


if __name__ == "__main__":
    try:
        asyncio.get_running_loop()
        print("Notebook detected: run `await main()` in a cell.")
    except RuntimeError:
        asyncio.run(main())


placeholder: rick.jpg from SALT-NLP/Design2Code
Notebook detected: run `await main()` in a cell.


In [ ]:
# quick test first (pages 0-7); set to None for the full run
END_INDEX = None
await main()

chromium 153.0.8010.12 | firefox 155.0, webkit 26.6
[9/484] d2c_008: 12 regression, 1 no_change, 2 benign
[10/484] d2c_009: 10 regression, 1 no_change, 2 benign
[11/484] d2c_010: 13 regression, 1 no_change, 2 benign
[12/484] d2c_011: 13 regression, 1 no_change, 2 benign
[13/484] d2c_012: 12 regression, 1 no_change, 2 benign
[14/484] d2c_013: 12 regression, 1 no_change, 2 benign
[15/484] d2c_014: 11 regression, 1 no_change, 2 benign
[16/484] d2c_015: 12 regression, 1 no_change, 2 benign
[17/484] d2c_016: 12 regression, 1 no_change, 2 benign
[18/484] d2c_017: 13 regression, 1 no_change, 2 benign
[19/484] d2c_018: 10 regression, 1 no_change, 2 benign
[20/484] d2c_019: 12 regression, 1 no_change, 2 benign
[21/484] d2c_020: 13 regression, 1 no_change, 2 benign
[22/484] d2c_021: 13 regression, 1 no_change, 2 benign
[23/484] d2c_022: 12 regression, 1 no_change, 2 benign
[24/484] d2c_023: 11 regression, 1 no_change, 2 benign
[25/484] d2c_024: 13 regression, 1 no_change, 2 benign
[26/484] d2c_0

In [17]:
!cd /content/drive/MyDrive/csc490 && zip -q -r -0 d2c_v3.zip d2c_v2
!cp /content/drive/MyDrive/csc490/d2c_v3.zip /content/drive/MyDrive/CSC490_data/